In [3]:
# Uncomment the line below if packages are not yet installed
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import sys
!{sys.executable} -m pip install langchain langchain-community langchain-openai langgraph faiss-cpu tavily-python streamlit tiktoken openai python-dotenv


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os
from dotenv import load_dotenv

load_dotenv()

API_Key = os.getenv("OPENAI_API_KEY")
Tavily_Key = os.getenv("TAVILY_API_KEY")

print("OpenAI key loaded:", bool(API_Key))
print("Tavily key loaded:", bool(Tavily_Key))

OpenAI key loaded: True
Tavily key loaded: True


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# -------------------------------------------------------
# SAMPLE TECHNICAL DOCUMENTATION
# In a real project, you'd load these from actual files:
#   loader = TextLoader('manual.txt')
#   docs = loader.load()
# -------------------------------------------------------
sample_docs = [
    Document(page_content="""
    How to Reset Your Password:
    1. Navigate to the login page at app.example.com
    2. Click 'Forgot Password' below the login button
    3. Enter your registered email address
    4. Check your email for a reset link (valid for 24 hours)
    5. Click the link and enter a new password (min 8 chars, 1 uppercase, 1 number)
    6. Log in with your new password
    Note: If no email arrives within 5 minutes, check your spam folder.
    """, metadata={"source": "user_manual.txt", "page": 1}),

    Document(page_content="""
    API Rate Limiting Policy:
    Our API enforces rate limits to ensure fair usage:
    - Free Tier: 100 requests/minute, 10,000 requests/day
    - Pro Tier: 1,000 requests/minute, 500,000 requests/day
    - Enterprise: Unlimited (contact sales)
    When the rate limit is exceeded, the API returns HTTP 429 (Too Many Requests).
    Implement exponential backoff: wait 1s, then 2s, then 4s between retries.
    Response headers include X-RateLimit-Remaining to track remaining quota.
    """, metadata={"source": "api_docs.txt", "page": 2}),

    Document(page_content="""
    Installation Guide for Windows:
    Requirements: Windows 10/11, 4GB RAM minimum, 2GB disk space
    Steps:
    1. Download the installer from downloads.example.com
    2. Right-click the .exe file and select 'Run as Administrator'
    3. Accept the license agreement
    4. Choose installation directory (default: C:\\Program Files\\ExampleApp)
    5. Click Install and wait for completion (~3 minutes)
    6. Launch from the Start Menu
    Troubleshooting: If you see 'DLL not found', install Visual C++ Redistributable from Microsoft.
    """, metadata={"source": "install_guide.txt", "page": 3}),

    Document(page_content="""
    Error Code Reference:
    ERR_001: Authentication failed - Check credentials or re-login
    ERR_002: Connection timeout - Check internet or firewall settings
    ERR_003: File not found - Verify the file path and permissions
    ERR_004: Insufficient storage - Free up at least 500MB disk space
    ERR_005: License expired - Contact support@example.com to renew
    ERR_429: Rate limit exceeded - Implement backoff and retry logic
    ERR_500: Server error - Our team is notified; retry in 5 minutes
    """, metadata={"source": "error_codes.txt", "page": 4}),

    Document(page_content="""
    Two-Factor Authentication (2FA) Setup:
    Enabling 2FA adds an extra security layer to your account.
    Supported methods: Authenticator App (recommended), SMS, Email OTP
    Setup steps:
    1. Go to Account Settings > Security
    2. Click 'Enable Two-Factor Authentication'
    3. Choose your preferred method
    4. For Authenticator App: scan the QR code with Google Authenticator or Authy
    5. Enter the 6-digit code to confirm
    6. Save your backup codes in a secure location!
    If you lose access to your 2FA device, use a backup code or contact support.
    """, metadata={"source": "security_guide.txt", "page": 5}),
]

# -------------------------------------------------------
# STEP 2A: Split documents into smaller chunks
# Why? LLMs have token limits, so we chunk docs and only
# send the most relevant pieces.
# chunk_size=500  → each chunk is ~500 characters
# chunk_overlap=50 → 50 chars overlap between chunks
#                     so context doesn't get cut off at edges
# -------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

chunks = text_splitter.split_documents(sample_docs)
print(f"✅ Created {len(chunks)} text chunks from {len(sample_docs)} documents")

# -------------------------------------------------------
# STEP 2B: Create embeddings and store in FAISS
# Embeddings = numerical representations of text meaning
# FAISS = a fast library for finding similar embeddings
# -------------------------------------------------------
embeddings = OpenAIEmbeddings()  # Uses OpenAI's text-embedding model

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # Fetch top 3 most relevant chunks

print("✅ FAISS vector store created and retriever configured!")

C:\Users\DEll\AppData\Local\Temp\ipykernel_9632\1613445124.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ Created 8 text chunks from 5 documents
✅ FAISS vector store created and retriever configured!


In [7]:
from typing import TypedDict, List, Optional
from langchain_core.documents import Document

class GraphState(TypedDict):
    """
    The shared state object that flows through the entire graph.
    Both agents READ from and WRITE to this object.
    
    Think of it like a baton in a relay race — each node updates it
    and passes it to the next.
    """
    original_query: str          # The raw question typed by the user
    optimized_query: str         # The rewritten, cleaner version of the query
    documents: List[Document]  # List of retrieved text chunks (from FAISS or web)
    generation: str              # The AI-generated answer
    loop_count: int              # Tracks how many correction loops have happened (max: 2)
    web_search_used: bool        # Whether the web fallback was triggered
    agent_trace: List[str]       # Log of decisions made (for UI display)

print("✅ GraphState defined with fields: original_query, optimized_query, documents, generation, loop_count, web_search_used, agent_trace")

✅ GraphState defined with fields: original_query, optimized_query, documents, generation, loop_count, web_search_used, agent_trace


In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.tavily_search import TavilySearchResults
import json

# -------------------------------------------------------
# Initialize the LLM (GPT-4o-mini is fast & cost-effective)
# temperature=0 means deterministic — same input → same output
# -------------------------------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Initialize Tavily web search tool (max 3 results per search)
tavily_tool = TavilySearchResults(max_results=3)

print("✅ LLM and Tavily web search tool initialized")

✅ LLM and Tavily web search tool initialized


C:\Users\DEll\AppData\Local\Temp\ipykernel_9632\2868419573.py:13: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=3)


In [9]:
# ==============================================================
# NODE 1: QUERY REWRITER
# Agent: Support Specialist
# 
# Purpose: Users often type vague or poorly-worded questions.
# This node uses an LLM to transform the raw query into a
# precise, keyword-rich search query that works better with
# the vector database.
#
# Example:
#   Input:  "i cant log in help"
#   Output: "login authentication failure troubleshooting steps password reset"
# ==============================================================

def rewrite_query_node(state: GraphState) -> dict:
    """
    Rewrites the user's raw query into a search-optimized version.
    This improves the quality of document retrieval from FAISS.
    """
    print("\n[Node 1] 🔄 Rewriting query...")

    query = state["original_query"]
    loop = state.get("loop_count", 0)
    trace = state.get("agent_trace", [])

    # Prompt that instructs the LLM to rewrite the query
    rewrite_prompt = ChatPromptTemplate.from_messages([
        ("system", """
You are a Search Query Optimizer for a technical support system.
Your job is to rewrite a user's vague or ambiguous tech support question 
into a precise, keyword-driven search query.

Rules:
- Focus on technical keywords (error codes, feature names, action verbs)
- Remove filler words like 'please', 'help', 'I need'
- Output ONLY the optimized query — no explanations or punctuation

Examples:
  Input:  "my app wont open after update"
  Output: "application startup failure post-update troubleshooting crash fix"

  Input:  "how do i change my email"
  Output: "update change email address account settings steps"
"""),
        ("human", "Rewrite this query for better search results:\n{query}")
    ])

    # Invoke the chain: prompt → LLM → get text response
    chain = rewrite_prompt | llm
    result = chain.invoke({"query": query})
    optimized = result.content.strip()

    print(f"   Original: '{query}'")
    print(f"   Optimized: '{optimized}'")

    trace_entry = f"🔄 Query rewritten (attempt {loop + 1}): '{optimized}'"
    trace.append(trace_entry)

    # Return updated state fields
    return {
        "optimized_query": optimized,
        "agent_trace": trace
    }


# ==============================================================
# NODE 2: DOCUMENT RETRIEVER
# Agent: Support Specialist
#
# Purpose: Uses the optimized query to search FAISS and retrieve
# the top 3 most relevant text chunks from our documentation.
# ==============================================================

def retrieve_docs_node(state: GraphState) -> dict:
    """
    Searches the FAISS vector store using the optimized query.
    Returns the top 3 most relevant document chunks.
    """
    print("\n[Node 2] 📚 Retrieving documents from FAISS...")

    trace = state.get("agent_trace", [])
    query = state["optimized_query"]

    # FAISS similarity search — finds chunks with meaning closest to our query
    docs = retriever.invoke(query)

    print(f"   Retrieved {len(docs)} document chunks")
    for i, doc in enumerate(docs):
        print(f"   Chunk {i+1}: {doc.page_content[:80]}...")

    trace.append(f"📚 Retrieved {len(docs)} chunks from local documentation")

    return {
        "documents": docs,
        "web_search_used": False,
        "agent_trace": trace
    }


# ==============================================================
# NODE 3: DOCUMENT GRADER (QA Agent)
# Agent: QA Engineer
#
# Purpose: The QA Agent reads each retrieved document and decides
# if it's actually relevant to the user's query.
# Output: Sets a flag 'all_relevant' = True/False in the state.
# ==============================================================

def grade_documents_node(state: GraphState) -> dict:
    """
    QA Agent grades each retrieved document for relevance.
    Uses structured JSON output: {"relevance_score": "yes" | "no"}
    If any document is irrelevant, routes to web search fallback.
    """
    print("\n[Node 3] 🔍 QA Agent grading documents...")

    query = state["original_query"]
    documents = state["documents"]
    trace = state.get("agent_trace", [])

    grading_prompt = ChatPromptTemplate.from_messages([
        ("system", """
You are a Document Relevance Grader for a tech support system.
Evaluate whether a retrieved document chunk is relevant to answering the user's question.

Output ONLY valid JSON in this exact format:
{{"relevance_score": "yes"}}
OR
{{"relevance_score": "no"}}

Rules:
- "yes" if the document contains information that helps answer the question
- "no" if the document is about a completely different topic
- Do not output anything else — no explanation, no extra text
"""),
        ("human", "User question: {query}\n\nDocument chunk:\n{document}")
    ])

    chain = grading_prompt | llm
    relevant_docs = []
    all_relevant = False

    for i, doc in enumerate(documents):
        # Ask the LLM to grade this specific document
        result = chain.invoke({"query": query, "document": doc.page_content})

        # Safely parse the JSON response
        try:
            score_data = json.loads(result.content.strip())
            score = score_data.get("relevance_score", "no")
        except json.JSONDecodeError:
            # If JSON parsing fails, default to 'no'
            score = "no"
            print(f"   ⚠️  JSON parsing failed for chunk {i+1}, defaulting to 'no'")

        if score == "yes":
            relevant_docs.append(doc)
            print(f"   Chunk {i+1}: ✅ RELEVANT")
        else:
            print(f"   Chunk {i+1}: ❌ IRRELEVANT")

    # If at least 1 relevant doc found, we're good to generate an answer
    all_relevant = len(relevant_docs) > 0

    if all_relevant:
        trace.append(f"✅ QA Agent: {len(relevant_docs)}/{len(documents)} docs are relevant — proceeding to generate")
    else:
        trace.append("⚠️ QA Agent: No relevant local docs found — triggering web search fallback")

    return {
        "documents": relevant_docs if all_relevant else documents,  # keep all if going to web
        "all_relevant": all_relevant,
        "agent_trace": trace
    }


# ==============================================================
# NODE 4: WEB SEARCH FALLBACK
# Agent: Support Specialist (using Tavily tool)
#
# Purpose: If local docs are irrelevant, search the live web
# using Tavily API and use those results as context instead.
# ==============================================================

def web_search_node(state: GraphState) -> dict:
    """
    Falls back to live web search when local documentation
    doesn't have a good answer. Uses Tavily API.
    """
    print("\n[Node 4] 🌐 Performing web search fallback...")

    query = state["optimized_query"]
    trace = state.get("agent_trace", [])

    # Tavily returns a list of {url, content} dicts
    results = tavily_tool.invoke({"query": query})

    # Convert web results into LangChain Document objects
    # so they're compatible with the rest of our pipeline
    web_docs = []
    for r in results:
        web_docs.append(
            Document(
                page_content=r.get("content", ""),
                metadata={"source": r.get("url", "web"), "type": "web_search"}
            )
        )

    print(f"   Found {len(web_docs)} web results")
    trace.append(f"🌐 Web search completed — retrieved {len(web_docs)} results from the internet")

    return {
        "documents": web_docs,
        "web_search_used": True,
        "agent_trace": trace
    }


# ==============================================================
# NODE 5: ANSWER GENERATOR
# Agent: Support Specialist
#
# Purpose: Takes all gathered context (docs) and generates a
# helpful, grounded technical support answer.
# The prompt explicitly tells the LLM to ONLY use provided docs.
# ==============================================================

def generate_answer_node(state: GraphState) -> dict:
    """
    Generates the final technical answer using the retrieved documents.
    The LLM is instructed to only use the provided context — no guessing.
    """
    print("\n[Node 5] ✍️  Generating answer...")

    query = state["original_query"]
    documents = state["documents"]
    trace = state.get("agent_trace", [])

    # Combine all document chunks into a single context string
    context = "\n\n---\n\n".join([doc.page_content for doc in documents])

    generation_prompt = ChatPromptTemplate.from_messages([
        ("system", """
You are an expert Tech Support Specialist at a global software company.
Your job is to provide clear, accurate, step-by-step technical support answers.

CRITICAL RULES:
1. ONLY use information from the provided context documents
2. If the context doesn't have enough info, say so clearly
3. Format your answer with numbered steps when applicable
4. Be concise but complete — no fluff or filler
5. Never invent information not present in the context

Context Documents:
{context}
"""),
        ("human", "Technical support question: {query}")
    ])

    chain = generation_prompt | llm
    result = chain.invoke({"context": context, "query": query})

    print(f"   Generated answer ({len(result.content)} chars)")
    trace.append("✍️ Support Agent generated a response")

    return {
        "generation": result.content,
        "agent_trace": trace
    }


# ==============================================================
# NODE 6: HALLUCINATION CHECKER (QA Agent)
# Agent: QA Engineer
#
# Purpose: The final quality gate. The QA Agent re-reads the
# generated answer and the source documents to verify:
#   - Is every claim in the answer supported by the docs?
#   - Did the AI make anything up?
# Output: {{"hallucination_check": "passed"} or {"hallucination_check": "failed"}
# ==============================================================

def hallucination_check_node(state: GraphState) -> dict:
    """
    QA Agent checks if the generated answer is grounded in the source docs.
    If hallucinations are detected, the loop_count is incremented
    and the graph re-routes back to the query rewriter.
    """
    print("\n[Node 6] 🧪 QA Agent checking for hallucinations...")

    generation = state["generation"]
    documents = state["documents"]
    loop_count = state.get("loop_count", 0)
    trace = state.get("agent_trace", [])

    context = "\n\n".join([doc.page_content for doc in documents])

    hallucination_prompt = ChatPromptTemplate.from_messages([
        ("system", """
You are a Hallucination Detection Judge for an AI support system.
Your job is to verify whether an AI-generated answer is fully supported by source documents.

Output ONLY valid JSON in this exact format:
{{"hallucination_check": "passed"}}
OR
{{"hallucination_check": "failed"}}

Rules for "passed":
- Every factual claim in the answer is directly supported by the source documents
- The answer doesn't add information not present in the sources
- The answer correctly addresses the question

Rules for "failed":
- Any claim appears to be invented or not traceable to the source documents
- The answer is vague, unhelpful, or off-topic
- The answer contradicts the source documents

Source Documents:
{context}
"""),
        ("human", "AI-Generated Answer:\n{generation}")
    ])

    chain = hallucination_prompt | llm
    result = chain.invoke({"context": context, "generation": generation})

    # Parse the JSON output safely
    try:
        check_data = json.loads(result.content.strip())
        check = check_data.get("hallucination_check", "failed")
    except json.JSONDecodeError:
        check = "failed"  # If parsing fails, be conservative and fail
        print("   ⚠️  JSON parsing failed, defaulting to 'failed'")

    if check == "passed":
        print("   ✅ PASSED — Answer is grounded and accurate")
        trace.append("✅ QA Agent: Hallucination check PASSED — answer is accurate and grounded")
        return {"loop_count": loop_count, "agent_trace": trace}
    else:
        new_loop_count = loop_count + 1
        print(f"   ❌ FAILED — Hallucination detected! Loop count: {new_loop_count}/2")
        trace.append(f"❌ QA Agent: Hallucination check FAILED (retry {new_loop_count}/2) — routing back to rewrite")
        return {"loop_count": new_loop_count, "agent_trace": trace}

print("✅ All node functions defined!")

✅ All node functions defined!


In [10]:
# ==============================================================
# ROUTING FUNCTION 1: After Document Grading
# 
# Decision: Are local docs relevant enough?
#   → YES: go to 'generate' (make an answer)
#   → NO:  go to 'web_search' (search the internet)
# ==============================================================

def route_after_grading(state: GraphState) -> str:
    """
    Checks the 'all_relevant' flag set by the grading node.
    Routes to either answer generation or web search.
    """
    if state.get("all_relevant", False):
        print("   ➡️  Routing to: GENERATE (docs are relevant)")
        return "generate"
    else:
        print("   ➡️  Routing to: WEB SEARCH (docs not relevant enough)")
        return "web_search"


# ==============================================================
# ROUTING FUNCTION 2: After Hallucination Check
#
# Decision: Is the answer good?
#   → PASS (loop_count unchanged): go to END
#   → FAIL (loop_count incremented): retry if < 2, else force END
#
# The loop_count check prevents infinite loops!
# ==============================================================

def route_after_hallucination_check(state: GraphState) -> str:
    """
    Checks the loop_count to decide whether to:
    - Accept the answer and END
    - Retry by going back to rewrite_query (if under limit)
    - Force END if we've retried too many times (prevents infinite loops)
    """
    loop_count = state.get("loop_count", 0)

    # Check if last hallucination check passed:
    # If loop_count stayed at its previous value, it passed.
    # We use a convention: if loop_count is same as before, it passed.
    # Actually we track this via the trace — simpler: check last trace entry
    last_trace = state.get("agent_trace", [""])[-1]

    if "PASSED" in last_trace:
        print("   ➡️  Routing to: END (QA passed!)")
        return "end"
    elif loop_count >= 2:
        # Hard safety limit: after 2 failed loops, accept the best answer we have
        print("   ➡️  Routing to: END (max retries reached, accepting current answer)")
        return "end"
    else:
        # Still under retry limit — go back and try again
        print(f"   ➡️  Routing to: REWRITE (retry {loop_count}/2)")
        return "rewrite"

print("✅ Routing functions defined!")

✅ Routing functions defined!


In [11]:
from langgraph.graph import StateGraph, END

# -------------------------------------------------------
# Create the StateGraph with our custom state type
# -------------------------------------------------------
workflow = StateGraph(GraphState)

# -------------------------------------------------------
# ADD NODES
# Each node is named and linked to its function
# -------------------------------------------------------
workflow.add_node("rewrite_query", rewrite_query_node)
workflow.add_node("retrieve_docs", retrieve_docs_node)
workflow.add_node("grade_docs", grade_documents_node)
workflow.add_node("web_search", web_search_node)
workflow.add_node("generate", generate_answer_node)
workflow.add_node("hallucination_check", hallucination_check_node)

# -------------------------------------------------------
# SET ENTRY POINT
# Where does the graph start?
# -------------------------------------------------------
workflow.set_entry_point("rewrite_query")

# -------------------------------------------------------
# ADD REGULAR (UNCONDITIONAL) EDGES
# These always go in one direction, no branching
# -------------------------------------------------------
workflow.add_edge("rewrite_query", "retrieve_docs")    # After rewrite, always retrieve
workflow.add_edge("retrieve_docs", "grade_docs")       # After retrieve, always grade
workflow.add_edge("web_search", "generate")            # After web search, always generate
workflow.add_edge("generate", "hallucination_check")   # After generate, always check

# -------------------------------------------------------
# ADD CONDITIONAL EDGES
# These branch based on routing function output
# -------------------------------------------------------

# After grading: either go to 'generate' or 'web_search'
workflow.add_conditional_edges(
    source="grade_docs",
    path=route_after_grading,
    path_map={
        "generate": "generate",
        "web_search": "web_search"
    }
)

# After hallucination check: either END or retry from 'rewrite_query'
workflow.add_conditional_edges(
    source="hallucination_check",
    path=route_after_hallucination_check,
    path_map={
        "end": END,
        "rewrite": "rewrite_query"  # This creates the CYCLE (self-correction loop)
    }
)

# -------------------------------------------------------
# COMPILE the graph into a runnable app
# -------------------------------------------------------
app = workflow.compile()

print("✅ LangGraph compiled successfully!")
print("\nGraph structure:")
print("  rewrite_query → retrieve_docs → grade_docs")
print("                                     ↓ (relevant)    ↓ (irrelevant)")
print("                                  generate  ←  web_search")
print("                                     ↓")
print("                               hallucination_check")
print("                                 ↓ pass     ↓ fail")
print("                                END    rewrite_query (loop, max 2x)")

✅ LangGraph compiled successfully!

Graph structure:
  rewrite_query → retrieve_docs → grade_docs
                                     ↓ (relevant)    ↓ (irrelevant)
                                  generate  ←  web_search
                                     ↓
                               hallucination_check
                                 ↓ pass     ↓ fail
                                END    rewrite_query (loop, max 2x)


In [12]:
def run_support_query(user_query: str):
    """
    Helper function to run a query through the entire multi-agent graph.
    Returns the final state with all information collected.
    """
    print("=" * 60)
    print(f"🎯 USER QUERY: {user_query}")
    print("=" * 60)

    # Initial state — only the user's query is known at start
    initial_state = {
        "original_query": user_query,
        "optimized_query": "",
        "documents": [],
        "generation": "",
        "loop_count": 0,
        "web_search_used": False,
        "agent_trace": []
    }

    # Run the graph! It will execute all nodes and routing automatically.
    final_state = app.invoke(initial_state)

    print("\n" + "=" * 60)
    print("📋 AGENT TRACE (Decision Log):")
    for step in final_state["agent_trace"]:
        print(f"  {step}")

    print("\n" + "=" * 60)

    print("💬 FINAL ANSWER:")
    print(final_state["generation"])
    print("=" * 60)

    return final_state


# -------------------------------------------------------
# TEST CASE 1: Query answerable from local docs
# -------------------------------------------------------
result1 = run_support_query("how do i reset my password i forgot it")

🎯 USER QUERY: how do i reset my password i forgot it

[Node 1] 🔄 Rewriting query...
   Original: 'how do i reset my password i forgot it'
   Optimized: 'reset forgotten password account recovery steps'

[Node 2] 📚 Retrieving documents from FAISS...
   Retrieved 3 document chunks
   Chunk 1: How to Reset Your Password:
    1. Navigate to the login page at app.example.com...
   Chunk 2: If you lose access to your 2FA device, use a backup code or contact support....
   Chunk 3: Two-Factor Authentication (2FA) Setup:
    Enabling 2FA adds an extra security l...

[Node 3] 🔍 QA Agent grading documents...
   Chunk 1: ✅ RELEVANT
   Chunk 2: ❌ IRRELEVANT
   Chunk 3: ❌ IRRELEVANT
   ➡️  Routing to: WEB SEARCH (docs not relevant enough)

[Node 4] 🌐 Performing web search fallback...
   Found 3 web results

[Node 5] ✍️  Generating answer...
   Generated answer (462 chars)

[Node 6] 🧪 QA Agent checking for hallucinations...
   ✅ PASSED — Answer is grounded and accurate
   ➡️  Routing to: END (QA pas

In [13]:
# -------------------------------------------------------
# TEST CASE 2: Query that might need web search
# -------------------------------------------------------
result2 = run_support_query("what are the best practices for securing kubernetes clusters in 2024")

🎯 USER QUERY: what are the best practices for securing kubernetes clusters in 2024

[Node 1] 🔄 Rewriting query...
   Original: 'what are the best practices for securing kubernetes clusters in 2024'
   Optimized: 'best practices securing Kubernetes clusters 2024'

[Node 2] 📚 Retrieving documents from FAISS...
   Retrieved 3 document chunks
   Chunk 1: If you lose access to your 2FA device, use a backup code or contact support....
   Chunk 2: Two-Factor Authentication (2FA) Setup:
    Enabling 2FA adds an extra security l...
   Chunk 3: API Rate Limiting Policy:
    Our API enforces rate limits to ensure fair usage:...

[Node 3] 🔍 QA Agent grading documents...
   Chunk 1: ❌ IRRELEVANT
   Chunk 2: ❌ IRRELEVANT
   Chunk 3: ❌ IRRELEVANT
   ➡️  Routing to: WEB SEARCH (docs not relevant enough)

[Node 4] 🌐 Performing web search fallback...
   Found 3 web results

[Node 5] ✍️  Generating answer...
   Generated answer (1214 chars)

[Node 6] 🧪 QA Agent checking for hallucinations...
   ✅ PASSED 

In [14]:
def run_with_streaming(user_query: str):
    """
    Runs the graph with streaming enabled.
    Each node's output is printed as soon as it completes.
    This is how the Streamlit UI gets live updates.
    """
    print(f"\n🚀 Running with streaming for: '{user_query}'\n")

    initial_state = {
        "original_query": user_query,
        "optimized_query": "",
        "documents": [],
        "generation": "",
        "loop_count": 0,
        "web_search_used": False,
        "agent_trace": []
    }

    # stream_mode="updates" gives us state changes after each node
    for step_output in app.stream(initial_state, stream_mode="updates"):
        # Each step_output is a dict like: {"node_name": updated_state_fields}
        node_name = list(step_output.keys())[0]
        node_data = step_output[node_name]

        print(f"📍 Completed node: [{node_name}]")

        # Show what changed in the state after this node
        for key, value in node_data.items():
            if key == "generation" and value:
                print(f"   generation: [Answer generated — {len(value)} chars]")
            elif key == "documents" and value:
                print(f"   documents: [{len(value)} chunks retrieved]")
            elif key == "agent_trace":
                if value:
                    print(f"   trace: {value[-1]}")  # Show only the latest trace entry
            else:
                print(f"   {key}: {value}")
        print()

# Test streaming
run_with_streaming("how do i set up two factor authentication")


🚀 Running with streaming for: 'how do i set up two factor authentication'


[Node 1] 🔄 Rewriting query...
   Original: 'how do i set up two factor authentication'
   Optimized: 'enable two-factor authentication setup guide'
📍 Completed node: [rewrite_query]
   optimized_query: enable two-factor authentication setup guide
   trace: 🔄 Query rewritten (attempt 1): 'enable two-factor authentication setup guide'


[Node 2] 📚 Retrieving documents from FAISS...
   Retrieved 3 document chunks
   Chunk 1: Two-Factor Authentication (2FA) Setup:
    Enabling 2FA adds an extra security l...
   Chunk 2: If you lose access to your 2FA device, use a backup code or contact support....
   Chunk 3: How to Reset Your Password:
    1. Navigate to the login page at app.example.com...
📍 Completed node: [retrieve_docs]
   documents: [3 chunks retrieved]
   web_search_used: False
   trace: 📚 Retrieved 3 chunks from local documentation


[Node 3] 🔍 QA Agent grading documents...
   Chunk 1: ✅ RELEVANT
   Chunk